# 08 – Runs vergleichen

Zweck: alle Trainings-Runs nebeneinander stellen – als Tabellen und Diagramme für die Auswertung und die
Abbildungen der Arbeit. Es wird **nichts neu berechnet**; gelesen werden nur die Dateien, die jeder Run
schreibt: `config.yaml`, `results.csv` (Val-Metriken pro Epoche), `test_metrics.json`, `robustness_metrics.json`.

- **Experiment** = Name der Run-Config (Ordner `weights/<Zeitstempel>_<Config-Name>/`; bei älteren Runs, deren
  Ordner nur den Zeitstempel trägt, aus der Zeile `Run config:` in `run.log`). Runs desselben Experiments mit
  verschiedenem `run.seed` werden in den Zusammenfassungen gemittelt: Mittelwert ± Standardabweichung über die Seeds, `n` = Anzahl Runs.
- **Beste Epoche** = die Epoche, deren Gewichte `best.pt` sind: höchste Val-mAP50-95 (Ultralytics-Fitness in dieser
  Version), bei Gleichstand die erste.
- **Vergleichbar** sind nur Runs mit gleichem Test-Set (sha256), gleichem Modell, gleicher Bildgröße, gleichen
  Trainings-Parametern und gleichen Daten. `check_comparable` warnt, wenn sich Runs in mehr als Augmentierung und Seed unterscheiden.
- Metriken: `precision`, `recall` (bei der besten F1-Schwelle der jeweiligen Gruppe), `mAP50`, `mAP50-95`,
  `false_alarms_per_frame` (Detektionen mit conf >= 0.25 auf Frames ohne Ziel).

In [ ]:
%run 00_utils.ipynb

In [ ]:
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

from config import WEIGHTS_DIR

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

## Einstellungen

In [ ]:
RUN_ROOTS = [WEIGHTS_DIR]   # Ordner mit Run-Ordnern; Dry-Runs dazu: [WEIGHTS_DIR, OUTPUT_DIR / "dryrun"]
INCLUDE = None              # Regex auf den Experiment-Namen, z.B. r"baseline|netd"; None = alle
ONLY_FINISHED = True        # nur Runs mit test_metrics.json (abgebrochene / laufende Runs weglassen)
METRIC = "mAP50"            # precision | recall | mAP50 | mAP50-95 | false_alarms_per_frame
GROUP = "all"               # Gruppe aus test_metrics.json, z.B. "size <=10px", "dataset CST-AntiUAV", "no target"
EXPORT_DIR = OUTPUT_DIR / "compare"   # CSV-Export der Tabellen (Abschnitt 6)

## 1. Runs laden

`find_runs` sucht in `RUN_ROOTS` alle Ordner `<YYYYMMDD-HHMMSS>[_<name>]` mit `config.yaml` und liest sie mit
`load_run`. Die Farben der Experimente werden aus **allen** gefundenen Experimenten vergeben (alphabetisch,
feste Reihenfolge), damit ein Experiment auch mit einem anderen `INCLUDE`-Filter dieselbe Farbe behält.

In [ ]:
RUN_DIR_RE = re.compile(r"^(\d{8}-\d{6})(?:_(.+))?$")
VAL_KEY = "metrics/mAP50-95(B)"   # Ultralytics-Fitness: best.pt = Epoche mit der höchsten Val-mAP50-95
SECTIONS = ["sota", "default", "edge_cases", "ir"]


def experiment_name(run_dir: Path) -> str:
    """Config-Name aus dem Ordnernamen; ältere Runs (nur Zeitstempel): erste 'Run config:'-Zeile in run.log."""
    m = RUN_DIR_RE.match(run_dir.name)
    if m and m.group(2):
        return m.group(2)
    log = run_dir / "run.log"
    if log.exists():
        with open(log, encoding="utf-8", errors="replace") as f:
            for line in f:
                if "Run config:" in line:
                    return Path(line.split("Run config:", 1)[1].split()[0]).stem
    return "?"


def enabled_augmentations(aug: dict) -> list[str]:
    """Eingeschaltete Augmentierungen als 'sektion.name(p=..)'; one_of-Gruppen als 'sektion.one_of[p=..](a|b)'."""
    out = []
    for section in aug.get("order") or SECTIONS:
        sec = aug.get(section) or {}
        if not sec.get("enabled", True):
            continue
        for item in sec.get("pipeline") or []:
            if "one_of" in item:
                g = item["one_of"] or {}
                members = [m["name"] for m in g.get("pipeline") or [] if m.get("enabled", True)]
                if g.get("enabled", True) and members:
                    out.append(f"{section}.one_of[p={g.get('p', 1.0)}]({'|'.join(members)})")
            elif item.get("enabled", True):
                out.append(f"{section}.{item['name']}(p={item.get('p', 1.0)})")
    return out


def _read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


def load_run(run_dir: Path) -> dict:
    """Alles, was der Vergleich von einem Run braucht (fehlende Dateien -> None)."""
    cfg = yaml.safe_load((run_dir / "config.yaml").read_text(encoding="utf-8")) or {}
    run_cfg, train, model = cfg.get("run") or {}, cfg.get("train") or {}, cfg.get("model") or {}
    curve = pd.read_csv(run_dir / "results.csv") if (run_dir / "results.csv").exists() else None
    if curve is not None:
        curve.columns = [c.strip() for c in curve.columns]
    best = curve.loc[curve[VAL_KEY].idxmax()] if curve is not None and len(curve) else None  # erste beste Epoche
    test = _read_json(run_dir / "test_metrics.json")
    return {
        "dir": run_dir,
        "run": run_dir.name,
        "experiment": experiment_name(run_dir),
        "seed": run_cfg.get("seed", 42),
        "model": model.get("weights"),
        "imgsz": model.get("imgsz", 640),
        "train": train,
        "data": {"datasets": cfg.get("datasets"),
                 "sampling": {k: v for k, v in (cfg.get("sampling") or {}).items() if k != "test"}},
        "epochs": train.get("epochs"),
        "augmentation": enabled_augmentations(cfg.get("augmentation") or {}),
        "dry_run": "dryrun" in run_dir.parts or bool(((test or {}).get("test_set") or {}).get("dry_run_subset")),
        "curve": curve,
        "epochs_trained": 0 if curve is None else len(curve),
        "best_epoch": None if best is None else int(best["epoch"]),
        "best_val": best,
        "test": test,
        "robustness": _read_json(run_dir / "robustness_metrics.json"),
    }


def find_runs(roots=None) -> list[dict]:
    """Alle Run-Ordner unter `roots` (Standard RUN_ROOTS), ältester zuerst."""
    runs = []
    for root in roots or RUN_ROOTS:
        root = Path(root)
        if root.exists():
            runs += [load_run(d) for d in sorted(root.iterdir())
                     if d.is_dir() and RUN_DIR_RE.match(d.name) and (d / "config.yaml").exists()]
    return runs


def select(runs, include=None, only_finished=True) -> list[dict]:
    return [r for r in runs
            if (not include or re.search(include, r["experiment"])) and (not only_finished or r["test"] is not None)]


# Kategoriale Farben in fester Reihenfolge (validierte Referenz-Palette, heller Hintergrund).
# Ab dem 9. Experiment: grau ("weitere") -> mit INCLUDE filtern statt neue Farben zu erfinden.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
OTHER = "#8a8985"


def color_map(runs) -> dict[str, str]:
    names = sorted({r["experiment"] for r in runs})
    if len(names) > len(PALETTE):
        print(f"Hinweis: {len(names)} Experimente, nur {len(PALETTE)} Farben - die übrigen sind grau. "
              "Für Diagramme mit INCLUDE filtern.")
    return {n: PALETTE[i] if i < len(PALETTE) else OTHER for i, n in enumerate(names)}

In [ ]:
all_runs = find_runs()
COLORS = color_map(all_runs)
runs = select(all_runs, INCLUDE, ONLY_FINISHED)
print(f"{len(all_runs)} Run-Ordner gefunden, {len(runs)} ausgewählt"
      + (f" (INCLUDE={INCLUDE!r})" if INCLUDE else "") + (" (nur fertige)" if ONLY_FINISHED else ""))

## 2. Übersicht und Vergleichbarkeit

`overview`: eine Zeile pro Run. `epochs` = trainiert/geplant; `early_stop` = vor der geplanten Epochenzahl
beendet (`patience`), nur bei fertigen Runs ohne Dry-Run. Liegt `best_epoch` nahe am Ende, hat das Modell
vermutlich noch nicht ausgelernt; liegt es früh, setzt danach Overfitting ein.

In [ ]:
def _test_group(r, group):
    return ((r["test"] or {}).get("groups") or {}).get(group) or {}


def overview(runs, metric=None) -> pd.DataFrame:
    metric = metric or METRIC
    rows = []
    for r in runs:
        b = r["best_val"]
        finished = r["test"] is not None
        rows.append({
            "run": r["run"],
            "experiment": r["experiment"],
            "seed": r["seed"],
            "model": r["model"],
            "epochs": f"{r['epochs_trained']}/{r['epochs']}",
            "best_epoch": r["best_epoch"],
            "early_stop": (finished and not r["dry_run"] and r["epochs"] is not None
                           and r["epochs_trained"] < r["epochs"]),
            "val mAP50-95 (best)": None if b is None else round(float(b[VAL_KEY]), 4),
            f"test {metric} (all)": _test_group(r, "all").get(metric),
            "test_set": (((r["test"] or {}).get("test_set") or {}).get("sha256") or "")[:12],
            "dry_run": r["dry_run"],
            "augmentation": ", ".join(r["augmentation"]) or "(keine)",
        })
    return pd.DataFrame(rows)


def check_comparable(runs) -> None:
    """Warnt, wenn sich die Runs in mehr als Augmentierung und Seed unterscheiden."""
    ignore_train = {"device", "workers"}  # ändern das Ergebnis nicht (workers: siehe Seed-Hinweis in der README)
    checks = {
        "Test-Set (sha256)": lambda r: ((r["test"] or {}).get("test_set") or {}).get("sha256"),
        "Test-Set-Teilmenge (Dry-Run)": lambda r: r["dry_run"],
        "Modell": lambda r: r["model"],
        "Bildgröße": lambda r: r["imgsz"],
        "Trainings-Parameter": lambda r: json.dumps({k: v for k, v in r["train"].items() if k not in ignore_train},
                                                    sort_keys=True, default=str),
        "Daten (datasets/sampling)": lambda r: json.dumps(r["data"], sort_keys=True, default=str),
        "Robustheits-Suite (sha256)": lambda r: ((r["robustness"] or {}).get("suite") or {}).get("sha256"),
    }
    ok = True
    for label, key in checks.items():
        values = {}
        for r in runs:
            v = key(r)
            if v is not None:
                values.setdefault(v, []).append(r["run"])
        if len(values) > 1:
            ok = False
            print(f"WARNUNG - {label} unterscheidet sich:")
            for v, names in values.items():
                print(f"    {str(v)[:100]}: {', '.join(names)}")
    dup = pd.Series([(r["experiment"], r["seed"]) for r in runs]).value_counts()
    for (exp, seed), n in dup[dup > 1].items():
        print(f"Hinweis: {n} Runs von '{exp}' mit demselben Seed {seed} (Wiederholung, kein neuer Seed).")
    if ok:
        print(f"OK: {len(runs)} Runs unterscheiden sich nur in Augmentierung und Seed.")

In [ ]:
display(overview(runs))
check_comparable(runs)

## 3. Test-Metriken

- `test_table`: alle Metriken einer Gruppe (`GROUP`), eine Zeile pro Run.
- `group_table`: eine Metrik (`METRIC`) für alle Gruppen – zeigt, ob eine Augmentierung z.B. nur großen Zielen hilft.
- `summary`: pro Experiment Mittelwert ± Standardabweichung über die Seeds (`n` = Anzahl Runs; bei `n` = 1 keine Streuung).

In [ ]:
TEST_COLS = ["frames", "targets", "precision", "recall", "mAP50", "mAP50-95", "false_alarms_per_frame"]


def test_table(runs, group=None) -> pd.DataFrame:
    group = group or GROUP
    rows = [{"run": r["run"], "experiment": r["experiment"], "seed": r["seed"],
             **{k: _test_group(r, group).get(k) for k in TEST_COLS}} for r in runs]
    return pd.DataFrame(rows).set_index("run")


def group_table(runs, metric=None) -> pd.DataFrame:
    metric = metric or METRIC
    rows = {r["run"]: {"experiment": r["experiment"], "seed": r["seed"],
                       **{g: v.get(metric) for g, v in ((r["test"] or {}).get("groups") or {}).items()}}
            for r in runs}
    return pd.DataFrame.from_dict(rows, orient="index")


def summary_long(runs, metric=None) -> pd.DataFrame:
    """Lange Tabelle: experiment, group, mean, std, n (über die Seeds)."""
    metric = metric or METRIC
    df = pd.DataFrame([{"experiment": r["experiment"], "seed": r["seed"], "group": g, "value": v.get(metric)}
                       for r in runs for g, v in ((r["test"] or {}).get("groups") or {}).items()])
    if df.empty:
        return pd.DataFrame(columns=["experiment", "group", "mean", "std", "n"])
    df = df.dropna(subset=["value"])
    return (df.groupby(["experiment", "group"], sort=False)["value"].agg(["mean", "std", "count"])
              .rename(columns={"count": "n"}).reset_index())


def summary(runs, metric=None, group=None) -> pd.DataFrame:
    metric, group = metric or METRIC, group or GROUP
    s = summary_long(runs, metric)
    s = s[s["group"] == group].drop(columns="group").set_index("experiment")
    s["seeds"] = pd.Series({e: sorted(r["seed"] for r in runs if r["experiment"] == e) for e in s.index})
    return s.round(4)

In [ ]:
print(f"Gruppe: {GROUP}")
display(test_table(runs))
print(f"Metrik: {METRIC}")
display(group_table(runs))
print(f"Pro Experiment ({METRIC}, Gruppe {GROUP}):")
display(summary(runs))

## 4. Robustheit

Nur Runs mit `robustness_metrics.json`. `value="delta"`: Änderung gegenüber den sauberen Testbildern
(negativ = das Modell verliert unter dieser Störung); `value="groups"`: der absolute Wert unter der Störung.
`robustness_summary` mittelt das Delta pro Experiment über die Seeds.

In [ ]:
def robustness_table(runs, metric=None, group=None, value="delta") -> pd.DataFrame:
    metric, group = metric or METRIC, group or GROUP
    rows = {}
    for r in runs:
        rob = r["robustness"]
        if not rob:
            continue
        row = {"experiment": r["experiment"], "seed": r["seed"]}
        if value == "groups":
            row["clean"] = (rob.get("clean", {}).get(group) or {}).get(metric)
        for name, t in rob.get("tests", {}).items():
            row[name] = (t.get(value, {}).get(group) or {}).get(metric)
        rows[r["run"]] = row
    return pd.DataFrame.from_dict(rows, orient="index")


def robustness_long(runs, metric=None, group=None) -> pd.DataFrame:
    """Lange Tabelle: experiment, test, mean, std, n des Deltas (über die Seeds)."""
    t = robustness_table(runs, metric, group, value="delta")
    if t.empty:
        return pd.DataFrame(columns=["experiment", "test", "mean", "std", "n"])
    df = t.drop(columns="seed").melt(id_vars="experiment", var_name="test", value_name="value").dropna()
    return (df.groupby(["experiment", "test"], sort=False)["value"].agg(["mean", "std", "count"])
              .rename(columns={"count": "n"}).reset_index())


def robustness_summary(runs, metric=None, group=None) -> pd.DataFrame:
    s = robustness_long(runs, metric, group)
    return s.pivot(index="experiment", columns="test", values="mean").round(4) if not s.empty else s

In [ ]:
rob = robustness_table(runs)
if rob.empty:
    print("Keine robustness_metrics.json in den ausgewählten Runs "
          "(run.robustness_suite setzen oder python src/main.py -e <run>).")
else:
    print(f"Delta {METRIC} (Gruppe {GROUP}) gegenüber sauber, pro Run:")
    display(rob)
    print("Mittel pro Experiment:")
    display(robustness_summary(runs))

## 5. Diagramme

- `plot_val_curves`: Val-mAP50-95 pro Epoche (das, woran `best.pt` gewählt wird); Punkt = beste Epoche.
  Mehrere Seeds eines Experiments haben dieselbe Farbe.
- `plot_groups`: `METRIC` pro Gruppe, ein Punkt pro Experiment (Mittel über die Seeds), Fehlerbalken =
  Standardabweichung (nur bei `n` > 1).
- `plot_robustness`: Delta pro Störung; die senkrechte Linie bei 0 = so gut wie auf sauberen Bildern.

Für Abbildungen der Arbeit: `fig.savefig(EXPORT_DIR / "name.pdf", bbox_inches="tight")`.

In [ ]:
INK, INK_2, GRID, AXIS = "#0b0b0b", "#52514e", "#e6e5e0", "#b5b4ae"


def _style(ax, title=None, xlabel=None, ylabel=None):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK_2, labelsize=9)
    if title:
        ax.set_title(title, color=INK, fontsize=11, loc="left")
    if xlabel:
        ax.set_xlabel(xlabel, color=INK_2, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, color=INK_2, fontsize=9)


def _legend(ax, experiments):
    handles = [plt.Line2D([], [], color=COLORS.get(e, OTHER), marker="o", linewidth=2, markersize=6, label=e)
               for e in experiments]
    ax.legend(handles=handles, frameon=False, fontsize=9, labelcolor=INK_2, loc="upper left",
              bbox_to_anchor=(1.01, 1.0))


def plot_val_curves(runs, key=VAL_KEY):
    runs = [r for r in runs if r["curve"] is not None and len(r["curve"])]
    if not runs:
        print("Keine results.csv gefunden.")
        return None
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for r in runs:
        c, color = r["curve"], COLORS.get(r["experiment"], OTHER)
        ax.plot(c["epoch"], c[key], color=color, linewidth=2, alpha=0.9)
        b = r["best_val"]
        ax.plot([b["epoch"]], [b[key]], marker="o", markersize=6, color=color,
                markeredgecolor="white", markeredgewidth=1.5, linestyle="none")
    _style(ax, f"Validierung: {key} pro Epoche (Punkt = beste Epoche = best.pt)", "Epoche", key)
    _legend(ax, sorted({r["experiment"] for r in runs}))
    fig.tight_layout()
    return fig


def _dot_plot(long_df, row_col, title, xlabel, zero_line=False):
    """Ein Punkt pro (Zeile, Experiment), Fehlerbalken = Standardabweichung; Experimente leicht versetzt."""
    if long_df.empty:
        print("Keine Daten.")
        return None
    rows = list(dict.fromkeys(long_df[row_col]))
    exps = sorted(long_df["experiment"].unique())
    fig, ax = plt.subplots(figsize=(9, 0.45 * len(rows) * max(1, len(exps) / 2) + 1.2))
    offsets = np.linspace(-0.3, 0.3, len(exps)) if len(exps) > 1 else [0.0]
    for off, e in zip(offsets, exps):
        d = long_df[long_df["experiment"] == e]
        y = [rows.index(v) + off for v in d[row_col]]
        ax.errorbar(d["mean"], y, xerr=d["std"].where(d["n"] > 1).fillna(0.0), fmt="o", markersize=6, linewidth=2,
                    capsize=0, color=COLORS.get(e, OTHER), markeredgecolor="white", markeredgewidth=1.5)
    if zero_line:
        ax.axvline(0, color=AXIS, linewidth=1)
    ax.set_yticks(range(len(rows)), rows)
    ax.invert_yaxis()
    _style(ax, title, xlabel)
    _legend(ax, exps)
    fig.tight_layout()
    return fig


def plot_groups(runs, metric=None, groups=None):
    metric = metric or METRIC
    s = summary_long(runs, metric)
    if groups is not None:
        s = s[s["group"].isin(groups)]
    return _dot_plot(s, "group", f"Test: {metric} pro Gruppe (Mittel über Seeds ± Std)", metric)


def plot_robustness(runs, metric=None, group=None):
    metric, group = metric or METRIC, group or GROUP
    return _dot_plot(robustness_long(runs, metric, group), "test",
                     f"Robustheit: Delta {metric} gegenüber sauber (Gruppe {group})", f"Delta {metric}",
                     zero_line=True)

In [ ]:
plot_val_curves(runs);

In [ ]:
plot_groups(runs);

In [ ]:
plot_robustness(runs);

## 6. Export

Schreibt die Tabellen als CSV nach `EXPORT_DIR` (z.B. für die Tabellen der Arbeit).

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
tables = {
    "overview": overview(runs),
    f"test_{GROUP}": test_table(runs),
    f"groups_{METRIC}": group_table(runs),
    f"summary_{METRIC}_{GROUP}": summary(runs),
    f"robustness_delta_{METRIC}_{GROUP}": robustness_table(runs),
}
for name, df in tables.items():
    path = EXPORT_DIR / f"{re.sub(r'[^A-Za-z0-9._-]+', '_', name)}.csv"
    df.to_csv(path)
    print(f"{path}  ({len(df)} Zeilen)")